In [ ]:
from ensemblr.ensemblr.monte_carlo.py import mc_path_optimisation

#### run MC simulated annealing #####

reproducible_mc = True  # reproducible = same set of seeds, otherwise seeds are randomised
mc_n_runs = 100

if reproducible_mc == True:
    with Pool(processes=num_processes) as pool:
        mc_runs = list(tqdm(pool.imap(mc_path_optimisation, range(mc_n_runs)), total=mc_n_runs))
else: 
    seeds = np.random.randint(low=0, high=1e3, size=mc_n_runs)  # Create a list of random seeds
    with Pool(processes=num_processes) as pool:
        mc_runs = list(tqdm(pool.imap(mc_path_optimisation, zip(range(mc_n_runs), seeds)), total=mc_n_runs))

# add the final energies and final path structures to a dataframe
mc_runs_df = pd.DataFrame(mc_runs, columns=['energy', 'path', 'path structures', 'relaxation energies'])
mc_runs_df = mc_runs_df.sort_values(by=['energy']) # NB this should ensure you write out the best path below

# write out the best run to a multistate pdb file 
with mda.Writer(base_directory + 'best_run.pdb', u.atoms.n_atoms) as W:
    # NB: head(1) when sorted gets the structures of the lowest energy path
    for structure in mc_runs_df.head(1)['path structures'].values[0]:
        u = mda.Universe(structure_directory + structure, structure_directory + structure)
        u.atoms.residues.resids += resid_offset
        if structure == 'ref_outward.pdb' or structure == 'ref_inward.pdb': 
            u.atoms.segments.segids = 'A'
            u.atoms.chainIDs = 'A'
        W.write(u.select_atoms('protein'))

##### plotting #####

# result and relaxation
display(mc_runs_df[['energy', 'path structures']].head(3))
for i in range(1, 4):
    plt.plot(mc_runs_df.iloc[i]['relaxation energies'])
plt.xlabel('MC step')
plt.ylabel('Energy')
plt.show()

# recalculate and plot energy vs window for debugging
energy_over_path = []
energy = {}
test = mc_runs_df.iloc[0]['path'] # get the "path" of the best run (in indices)
for i in range(mc_n_bins - 1):
        energy[i] = (rmsd_matrix[np.where(ensemble_df.index.values == test[i])[0][0], np.where(ensemble_df.index.values == test[i+1])[0][0]]**2)
        energy[i] = np.sqrt ( 1 * energy[i] )
plt.plot(energy.values(), label='best')
test = mc_runs_df.iloc[-1]['path']
for i in range(mc_n_bins - 1):
        energy[i] = (rmsd_matrix[np.where(ensemble_df.index.values == test[i])[0][0], np.where(ensemble_df.index.values == test[i+1])[0][0]]**2)
        energy[i] = np.sqrt ( 1 * energy[i] )
plt.plot(energy.values(), label='worst')
plt.legend()
plt.xlabel('point in path')
plt.ylabel('MC energy')
plt.show()
